<div align="center">
  <p><strong><a href="https://br.linkedin.com/in/isaac-maciel" target="_blank">Autor do Notebook: Isaac Maciel</a></strong></p>

<p align="center">
  <img src="https://github.com/IM-NOT-AI/IM-not-AI/assets/113378671/0a6e0c0f-fcdf-4ccc-af00-63e30836d180" alt="gif_isaac_nome" width="500">
</p>

In [ ]:
%reload_ext watermark
%watermark -a "Isaac Maciel" -v -m
%watermark --iversions

# Poda de Dados e Curadoria Determinística - Data Pruning

**Objetivo:** Extrair apenas os eixos de decisão clínica (regras, dosagens e fluxogramas) dos PDFs brutos, expurgando ruídos acadêmicos (metodologia, epidemiologia e referências). Esta etapa reduz a dimensionalidade do Corpus em até 80%, viabilizando a vetorização leve para Edge Computing.

### Mapa de Corte - Pruning Map - Diretrizes e Protocolos

Se jogarmos PDFs inteiros no algoritmo de vetorização, o modelo vai aprender sobre "Epidemiologia do infarto" e "Estatística de Mortalidade" em vez de focar no que importa para o Raspberry Pi: **"O que fazer agora para salvar o paciente"**.

Nós precisamos extirpar (cortar) tudo o que for texto introdutório, justificativa acadêmica, revisões sistemáticas e focarmos exclusivamente em **regras imperativas, fluxogramas, escores clínicos e dosagens.**

Abaixo está o seu **"Mapa de Corte"** exato para cada um dos 5 documentos dessa past: *_"diretrizes_e_protocolos"_*. Vamos abrir  usar um script Python com PyPDF2 e extrair apenas as páginas/seções listadas abaixo. Todo o resto será deletado.

`diretriz_sbc_angina_instavel.pdf`

**Excluir**

* Epidemiologia, Fisiopatologia, testes de longo prazo (Parte 3 inteira sobre mudança de estilo de vida, nutrição e reabilitação). O robô não precisa saber sobre dieta pós-alta na sala de emergência.

**Manter**

* **Páginas 206 a 210**
    * Estratificação de Risco e Fluxogramas na Emergência. (É aqui que a IA aprende a regra "Se o paciente tiver X, interne; se tiver Y, dê alta").

* **Páginas 211 a 218**
    * Condutas na Emergência. (Regras de oxigênio, analgesia, nitratos e bloqueadores de receptor P2Y12).

* **Páginas 220 a 226**
    * Fármacos Antiplaquetários e Antitrombínicos. (Crucial para o modelo aprender as dosagens exatas de Clopidogrel, Ticagrelor e Heparina).

<br>

`diretriz_sbc_fibrilacao_atrial.pd`

**Excluir**

* Capítulos 1 a 4 (Introdução, Epidemiologia, Mecanismos), e Capítulos 8 e 9 (Técnicas cirúrgicas de Ablação e Marcapasso). A IA não vai operar o paciente, ela vai dar a conduta imediata.

**Manter**

* **Páginas 28 a 44**
    * Algoritmos de Avaliação de Risco (CHA2DS2-VASc e HAS-BLED) e regras de Anticoagulação.

* **Páginas 50 a 57**
    * Tratamento do Ritmo ou da Frequência Cardíaca (Uso agudo de Amiodarona, Propafenona, Betabloqueadores e as regras de Cardioversão Elétrica/Química).

<br>

`diretriz_sbc_ressuscitacao_cardiopulmonar.pdf`

**Excluir**

* Capítulos 1, 16, 17, 18 e 20 (Epidemiologia, Simulação, Regulação Pré-hospitalar e Princípios Éticos/Legais).

**Manter - O coração do Algortimo**

* **Páginas 469 a 472** 
    * Terapias Elétricas (Indicações e Joules para desfibrilação).

* **Páginas 475 a 492**
    * Suporte Avançado de Vida (ACLS). (Este é o texto mais importante de todo o projeto. Ele contém as regras de Adrenalina, Amiodarona, FV/TV sem pulso e Assistolia).

* **Páginas 500 a 505**
    * Tratamento da Síndrome Coronariana Aguda na Emergência (Escores TIMI/GRACE e terapias de reperfusão).

* **Páginas 581 a 588**
    * Suporte Avançado em Insuficiência Cardíaca Aguda.

<br>

`diretriz_sus_insuficiencia_cardiaca.pdf`

**ALERTA DE ENGHERARIA DE DADOS**

* **Sumário de Evidências GRADE** (Análise Estatística de Risco Relativo - RR, Intervalo de Confiança).
    * **AÇÃO** DELETAR TODAS essas tabelas de metodologia científica. Se deixar isso, a matriz TF-IDF vai ser poluída com tokens matemáticos inúteis como "IC95%", "Follow-up", "Viés" e "Grau de Evidência".

    * **O que buscar neste PDF:** Devemos ignorar essas páginas e buscar, no início ou no fim do documento, a seção "Algoritmo de Tratamento" ou "Recomendações Terapêuticas" (onde o SUS diz: "Administrar Furosemida 40mg EV" ou "Iniciar IECA"). Extrair apenas essas páginas de conduta clínica direta.

<br>

`protocolo_sus_sindrome_coronariana.pdf`

**ALERTA DE ENGHERARIA DE DADOS**

* Contém as Siglas, Introdução e Metodologia.

    * **AÇÃO:** Deletar páginas 1 a 4. As siglas já serão resolvidas pelo NLP de qualquer forma.
    * **O que buscar neste PDF:** Iremos direto para as seções de Critérios Diagnósticos, Classificação de Risco e Fluxo de Tratamento (Medicamentos e Tempo-Alvo).

#### Imports

In [7]:
import os
from PyPDF2 import PdfReader, PdfWriter

In [8]:
# ==============================================================================
# CLASSE ORQUESTRADORA DE PODA DE DADOS (POO)
# ==============================================================================
class OrquestradorPodaPDF:
    """
    Classe responsavel por encapsular e distribuir a logica de extracao
    e poda de dados em documentos PDF, garantindo a reutilizacao de codigo
    para multiplos dominios clinicos do Data Lake.
    """
    def __init__(self, diretorio_origem, diretorio_destino):
        self.diretorio_origem = diretorio_origem
        self.diretorio_destino = diretorio_destino
        self.arquivos_processados = 0
        
        # Garante a integridade da arvore de diretorios (Data Lake)
        os.makedirs(self.diretorio_destino, exist_ok=True)

    def processar_documento(self, nome_arquivo, regras):
        """
        Aplica as regras de corte a um documento especifico e o salva no diretorio de destino.
        """
        caminho_entrada = os.path.join(self.diretorio_origem, nome_arquivo)
        caminho_saida = os.path.join(self.diretorio_destino, nome_arquivo.replace(".pdf", "_filtrado.pdf"))
        
        # Validacao de I/O
        if not os.path.exists(caminho_entrada):
            print(f"[ERRO] Arquivo ausente na origem: {caminho_entrada}")
            return False
            
        print(f"[PROCESSANDO] {nome_arquivo}...")
        
        try:
            leitor_pdf = PdfReader(caminho_entrada)
            escritor_pdf = PdfWriter()
            total_paginas = len(leitor_pdf.pages)
            paginas_extraidas = 0
            
            offset = regras.get("offset", 0)
            
            for intervalo in regras.get("ranges", []):
                inicio = intervalo[0]
                fim = intervalo[1] if intervalo[1] is not None else total_paginas
                
                # Matematica de deslocamento do indice da revista vs. indice do PDF
                idx_inicio = max(0, (inicio - 1) + offset)
                idx_fim = min(total_paginas, (fim) + offset)
                
                for num_pagina in range(idx_inicio, idx_fim):
                    # Seguranca contra Index Out of Bounds
                    if num_pagina < total_paginas:
                        escritor_pdf.add_page(leitor_pdf.pages[num_pagina])
                        paginas_extraidas += 1
                        
            # Serializa o novo PDF enxuto no disco
            with open(caminho_saida, "wb") as arquivo_saida:
                escritor_pdf.write(arquivo_saida)
                
            taxa_reducao = 100 - ((paginas_extraidas / total_paginas) * 100)
            print(f"  -> Sucesso! Extraidas {paginas_extraidas} paginas de {total_paginas} originais.")
            print(f"  -> Reducao Dimensional Focada: {taxa_reducao:.1f}% do ruido eliminado.")
            self.arquivos_processados += 1
            return True
            
        except Exception as erro:
            print(f"  -> [FALHA CRITICA] Erro ao processar {nome_arquivo}. Detalhe: {erro}")
            return False

    def executar_pipeline(self, dicionario_regras, nome_pipeline="DOMINIO GENERICO"):
        """
        Metodo principal que distribui as tarefas de leitura iterando sobre o dicionario.
        """
        print(f"=== INICIANDO PIPELINE DE DESTILACAO: {nome_pipeline} ===\n")
        print(f"[I/O] Diretorio Ingestao: {self.diretorio_origem}")
        print(f"[I/O] Diretorio Exportacao: {self.diretorio_destino}\n")
        
        for arquivo, regras in dicionario_regras.items():
            self.processar_documento(arquivo, regras)
            
        print(f"\n[AUDITORIA] Processamento Finalizado. {self.arquivos_processados} arquetipos purificados e salvos.")

In [9]:
# ------------------------------------------------------------------------------
# DIRETRIZES E PROTOCOLOS
# ------------------------------------------------------------------------------
dir_raw_diretrizes = os.path.join("..", "data", "raw", "nlp")
dir_processed_diretrizes = os.path.join("..", "data", "processed", "nlp", "pruned_pdfs")

regras_diretrizes = {
    "diretriz_sbc_angina_instavel.pdf": {
        "ranges": [(206, 210), (211, 218), (220, 226)],
        "offset": -180
    },
    "diretriz_sbc_fibrilacao_atrial.pdf": {
        "ranges": [(28, 44), (50, 57)],
        "offset": 0 
    },
    "diretriz_sbc_ressuscitacao_cardiopulmonar.pdf": {
        "ranges": [(469, 472), (475, 492), (500, 505), (581, 588)],
        "offset": -448
    },
    "protocolo_sus_sindrome_coronariana.pdf": {
        "ranges": [(5, None)], 
        "offset": 0
    },
    "diretriz_sus_insuficiencia_cardiaca.pdf": {
        "ranges": [(15, 25)], 
        "offset": 0
    }
}

# Instancia o objeto e roda o motor
orquestrador_diretrizes = OrquestradorPodaPDF(dir_raw_diretrizes, dir_processed_diretrizes)
orquestrador_diretrizes.executar_pipeline(regras_diretrizes, "DIRETRIZES E PROTOCOLOS")

=== INICIANDO PIPELINE DE DESTILACAO: DIRETRIZES E PROTOCOLOS ===

[I/O] Diretorio Ingestao: ..\data\raw\nlp
[I/O] Diretorio Exportacao: ..\data\processed\nlp\pruned_pdfs

[PROCESSANDO] diretriz_sbc_angina_instavel.pdf...
  -> Sucesso! Extraidas 20 paginas de 84 originais.
  -> Reducao Dimensional Focada: 76.2% do ruido eliminado.
[PROCESSANDO] diretriz_sbc_fibrilacao_atrial.pdf...
  -> Sucesso! Extraidas 25 paginas de 107 originais.
  -> Reducao Dimensional Focada: 76.6% do ruido eliminado.
[PROCESSANDO] diretriz_sbc_ressuscitacao_cardiopulmonar.pdf...
  -> Sucesso! Extraidas 36 paginas de 215 originais.
  -> Reducao Dimensional Focada: 83.3% do ruido eliminado.
[PROCESSANDO] protocolo_sus_sindrome_coronariana.pdf...
  -> Sucesso! Extraidas 42 paginas de 46 originais.
  -> Reducao Dimensional Focada: 8.7% do ruido eliminado.
[PROCESSANDO] diretriz_sus_insuficiencia_cardiaca.pdf...
  -> Sucesso! Extraidas 11 paginas de 108 originais.
  -> Reducao Dimensional Focada: 89.8% do ruido elim

### Mapa de Corte - Pruning Map - Relatos Clinicos


* **Alerta de Engenharia de Dados (Alta Granularidade):**
  Relatos de caso são documentos curtos. O objetivo da poda aqui é extrair apenas a folha inicial que contém a anamnese e os achados laboratoriais, descartando as páginas finais de revisão de literatura e bibliografia.

**Arquivos e Cortes:**
* **relato_caso_assistencia_circulatoria_chagas.pdf** 
    * Manter páginas 112 e 113.
* **relato_caso_disfuncao_cardiaca_quimioterapia.pdf** 
    * Manter páginas 3 e 4 (Onde reside a seção Resultados/Caso).
* **relato_caso_fibrilacao_ventricular_cardiotoxicidade.pdf**
    * Manter páginas 1 e 2.
* **relato_caso_iamcsst_ruptura_parede_livre.pdf**
    * Manter páginas 25 e 26.
* **relato_caso_ic_descompensada_arbovirose.pdf**
    * Manter páginas 19 e 20.

* **relato_caso_miocardite_coinfeccao_arboviroses.pdf**
    * Manter páginas 783 e 784.
* **relato_caso_miocardite_covid19.pdf**
    * Manter páginas 839 a 841.
* **relato_caso_takotsubo.pdf** 
    * Manter páginas 1 e 2.
    
*(Nota: O arquivo do congresso de arritmias foi removido por reprovação na auditoria de qualidade dos dados).*

In [10]:
# ------------------------------------------------------------------------------
# RELATOS CLINICOS
# ------------------------------------------------------------------------------
dir_raw_relatos = os.path.join("..", "data", "raw", "nlp")
dir_processed_relatos = os.path.join("..", "data", "processed", "nlp", "pruned_pdfs")

regras_relatos = {
    "relato_caso_assistencia_circulatoria_chagas.pdf": {
        "ranges": [(112, 113)],
        "offset": -111  
    },
    "relato_caso_disfuncao_cardiaca_quimioterapia.pdf": {
        "ranges": [(3, 4)],
        "offset": 0     
    },
    "relato_caso_fibrilacao_ventricular_cardiotoxicidade.pdf": {
        "ranges": [(1, 2)],
        "offset": 0     
    },
    "relato_caso_iamcsst_ruptura_parede_livre.pdf": {
        "ranges": [(25, 26)],
        "offset": -24   
    },
    "relato_caso_ic_descompensada_arbovirose.pdf": {
        "ranges": [(19, 20)],
        "offset": -18   
    },
    "relato_caso_miocardite_coinfeccao_arboviroses.pdf": {
        "ranges": [(783, 784)],
        "offset": -782
    },
    "relato_caso_miocardite_covid19.pdf": {
        "ranges": [(839, 841)],
        "offset": -838
    },
    "relato_caso_takotsubo.pdf": {
        "ranges": [(1, 2)],
        "offset": 0
    }
}

# Instancia um NOVO objeto e roda o motor
orquestrador_relatos = OrquestradorPodaPDF(dir_raw_relatos, dir_processed_relatos)
orquestrador_relatos.executar_pipeline(regras_relatos, "RELATOS CLINICOS")

=== INICIANDO PIPELINE DE DESTILACAO: RELATOS CLINICOS ===

[I/O] Diretorio Ingestao: ..\data\raw\nlp
[I/O] Diretorio Exportacao: ..\data\processed\nlp\pruned_pdfs

[PROCESSANDO] relato_caso_assistencia_circulatoria_chagas.pdf...
  -> Sucesso! Extraidas 2 paginas de 3 originais.
  -> Reducao Dimensional Focada: 33.3% do ruido eliminado.
[PROCESSANDO] relato_caso_disfuncao_cardiaca_quimioterapia.pdf...
  -> Sucesso! Extraidas 2 paginas de 7 originais.
  -> Reducao Dimensional Focada: 71.4% do ruido eliminado.
[PROCESSANDO] relato_caso_fibrilacao_ventricular_cardiotoxicidade.pdf...
  -> Sucesso! Extraidas 2 paginas de 4 originais.
  -> Reducao Dimensional Focada: 50.0% do ruido eliminado.
[PROCESSANDO] relato_caso_iamcsst_ruptura_parede_livre.pdf...
  -> Sucesso! Extraidas 2 paginas de 6 originais.
  -> Reducao Dimensional Focada: 66.7% do ruido eliminado.
[PROCESSANDO] relato_caso_ic_descompensada_arbovirose.pdf...
  -> Sucesso! Extraidas 2 paginas de 4 originais.
  -> Reducao Dimension

### Mapa de Corte - Pruning Map - Revisões Acadêmicas


**Alerta de Engenharia de Dados (Controle de Ruído Bibliográfico):** Artigos de revisão possuem uma carga altíssima de referências bibliográficas ao final do documento. O motor de NLP não pode aprender nomes de autores ou anos de publicação. A poda nesta categoria visa preservar os resumos, tabelas de risco e conclusões clínicas, expurgando as páginas finais.

**Arquivos e Cortes:**
* **revisao_ceramidas_plasmaticas_risco_cardiovascular.pdf** 
    * Manter páginas 768 a 771 (Foco na tabela de estratificação).
* **revisao_estrogenio_obesidade_insuficiencia_cardiaca.pdf**
    * Manter páginas 1191 a 1194.
* **revisao_fibrilacao_atrial_pacientes_cancer.pdf**
    * Manter páginas 328 a 331.
* **revisao_fibrilacao_atrial.pdf**
    * Manter páginas 3935 a 3937.
* **revisao_gdf15_biomarcador_doencas_cardiovasculares.pdf**
    * Manter páginas 494 a 500 (Contém a tabela vital de pontos de corte do GDF-15).
* **revisao_indices_hematologicos_inflamatorios_mortalidade.pdf**
    * Manter páginas 1 a 4.
* **revisao_mirnas_fisiopatologia_cardiovascular.pdf**
    * Manter páginas 738 a 741.
* **revisao_sindrome_cardiorrenal_criterios_prognostico.pdf**
    * Manter páginas 127 a 130.
* **relato_caso_disseccao_aortica_stanford.pdf**
    * Manter páginas 773 a 775 (Movido de relatos para revisões após auditoria de qualidade de dados).

In [11]:
# ==============================================================================
# REVISOES ACADEMICAS
# ==============================================================================
dir_raw_revisoes = os.path.join("..", "data", "raw", "nlp")
dir_processed_revisoes = os.path.join("..", "data", "processed", "nlp", "pruned_pdfs")

regras_revisoes = {
    "revisao_ceramidas_plasmaticas_risco_cardiovascular.pdf": {
        "ranges": [(768, 771)],
        "offset": -767
    },
    "revisao_estrogenio_obesidade_insuficiencia_cardiaca.pdf": {
        "ranges": [(1191, 1194)],
        "offset": -1190
    },
    "revisao_fibrilacao_atrial_pacientes_cancer.pdf": {
        "ranges": [(328, 331)],
        "offset": -327
    },
    "revisao_fibrilacao_atrial.pdf": {
        "ranges": [(3935, 3937)],
        "offset": -3934
    },
    "revisao_gdf15_biomarcador_doencas_cardiovasculares.pdf": {
        "ranges": [(494, 500)],
        "offset": -493
    },
    "revisao_indices_hematologicos_inflamatorios_mortalidade.pdf": {
        "ranges": [(1, 4)],
        "offset": 0
    },
    "revisao_mirnas_fisiopatologia_cardiovascular.pdf": {
        "ranges": [(738, 741)],
        "offset": -737
    },
    "revisao_sindrome_cardiorrenal_criterios_prognostico.pdf": {
        "ranges": [(127, 130)],
        "offset": -126
    }
}

# Instancia o objeto para o novo dominio e roda o motor
orquestrador_revisoes = OrquestradorPodaPDF(dir_raw_revisoes, dir_processed_revisoes)
orquestrador_revisoes.executar_pipeline(regras_revisoes, "REVISOES ACADEMICAS")

=== INICIANDO PIPELINE DE DESTILACAO: REVISOES ACADEMICAS ===

[I/O] Diretorio Ingestao: ..\data\raw\nlp
[I/O] Diretorio Exportacao: ..\data\processed\nlp\pruned_pdfs

[PROCESSANDO] revisao_ceramidas_plasmaticas_risco_cardiovascular.pdf...
  -> Sucesso! Extraidas 4 paginas de 10 originais.
  -> Reducao Dimensional Focada: 60.0% do ruido eliminado.
[PROCESSANDO] revisao_estrogenio_obesidade_insuficiencia_cardiaca.pdf...
  -> Sucesso! Extraidas 4 paginas de 11 originais.
  -> Reducao Dimensional Focada: 63.6% do ruido eliminado.
[PROCESSANDO] revisao_fibrilacao_atrial_pacientes_cancer.pdf...
  -> Sucesso! Extraidas 4 paginas de 14 originais.
  -> Reducao Dimensional Focada: 71.4% do ruido eliminado.
[PROCESSANDO] revisao_fibrilacao_atrial.pdf...
  -> Sucesso! Extraidas 3 paginas de 9 originais.
  -> Reducao Dimensional Focada: 66.7% do ruido eliminado.
[PROCESSANDO] revisao_gdf15_biomarcador_doencas_cardiovasculares.pdf...
  -> Sucesso! Extraidas 7 paginas de 7 originais.
  -> Reducao Di

### Mapa de Corte - Pruning Map - Farmacologia e Bulas



**Alerta de Engenharia de Dados (Filtro de Ruído Regulatório e Otimização TF-IDF):**
Bulas profissionais possuem um formato rígido exigido pela ANVISA. O "Sinal clínico" está concentrado no início do documento. As páginas finais formam a "Cauda Burocrática" (CNPJ, Farmacêutico Responsável, telefones de SAC, reciclagem de embalagem). Se não extrairmos apenas o núcleo duro, a matriz TF-IDF do modelo será inundada por palavras administrativas, diluindo o peso dos termos clínicos vitais e consumindo processamento inútil no Raspberry Pi.

**Excluir**
* Dados de armazenamento, aspecto físico da embalagem e regras de descarte do material.
* Estudos toxicológicos pré-clínicos (testes em cobaias).
* Informações legais, SAC e registros de responsáveis técnicos (CRF, CNPJ).

**Manter**
* Indicações e Contraindicações: O "freio de segurança" do algoritmo para não sugerir terapias letais.
* Posologia e Interações Medicamentosas: Regras numéricas exatas de diluição e dosagem de ataque.

**Arquivos e Cortes:**
* **bula_profissional_alteplase.pdf**
    * Manter páginas 1 a 10. (Garante a extração das janelas de tempo e dosagem para IAM/AVC. Exclui as referências de fabricação e SAC).
* **bula_profissional_amiodarona.pdf**
    * Manter páginas 1 a 10. (Preserva a farmacodinâmica e as doses de ataque em bomba de infusão para arritmias. Corta o extenso texto sobre validade e registro).
* **bula_profissional_clopidogrel.pdf**
    * Manter páginas 1 a 12. (Retém as interações medicamentosas vitais e ajustes de dose de ataque de 300mg. Expulga a cauda de burocracia governamental final).
* **bula_profissional_enoxaparina.pdf**
    * Manter páginas 1 a 8. (Foca estritamente no cálculo de dose por peso e clearance de creatinina. Remove detalhes logísticos sobre como descartar a seringa plástica).
* **bula_profissional_noradrenalina.pdf**
    * Manter páginas 1 a 8. (Captura as taxas de infusão contínua vasoativa e diluição padrão. Elimina as seções legais e informações da empresa farmacêutica).

In [12]:
# ==============================================================================
# FARMACOLOGIA E BULAS
# ==============================================================================
dir_raw_bulas = os.path.join("..", "data", "raw", "nlp")
dir_processed_bulas = os.path.join("..", "data", "processed", "nlp", "pruned_pdfs")

regras_bulas = {
    "bula_profissional_alteplase.pdf": {
        "ranges": [(1, 10)],
        "offset": 0
    },
    "bula_profissional_amiodarona.pdf": {
        "ranges": [(1, 10)],
        "offset": 0
    },
    "bula_profissional_clopidogrel.pdf": {
        "ranges": [(1, 12)],
        "offset": 0
    },
    "bula_profissional_enoxaparina.pdf": {
        "ranges": [(1, 8)],
        "offset": 0
    },
    "bula_profissional_noradrenalina.pdf": {
        "ranges": [(1, 8)],
        "offset": 0
    }
}

# Instancia o objeto para o ultimo dominio e roda o motor
orquestrador_bulas = OrquestradorPodaPDF(dir_raw_bulas, dir_processed_bulas)
orquestrador_bulas.executar_pipeline(regras_bulas, "FARMACOLOGIA E BULAS")

=== INICIANDO PIPELINE DE DESTILACAO: FARMACOLOGIA E BULAS ===

[I/O] Diretorio Ingestao: ..\data\raw\nlp
[I/O] Diretorio Exportacao: ..\data\processed\nlp\pruned_pdfs

[PROCESSANDO] bula_profissional_alteplase.pdf...
  -> Sucesso! Extraidas 10 paginas de 11 originais.
  -> Reducao Dimensional Focada: 9.1% do ruido eliminado.
[PROCESSANDO] bula_profissional_amiodarona.pdf...
  -> Sucesso! Extraidas 10 paginas de 29 originais.
  -> Reducao Dimensional Focada: 65.5% do ruido eliminado.
[PROCESSANDO] bula_profissional_clopidogrel.pdf...
  -> Sucesso! Extraidas 12 paginas de 23 originais.
  -> Reducao Dimensional Focada: 47.8% do ruido eliminado.
[PROCESSANDO] bula_profissional_enoxaparina.pdf...
  -> Sucesso! Extraidas 8 paginas de 9 originais.
  -> Reducao Dimensional Focada: 11.1% do ruido eliminado.
[PROCESSANDO] bula_profissional_noradrenalina.pdf...
  -> Sucesso! Extraidas 8 paginas de 9 originais.
  -> Reducao Dimensional Focada: 11.1% do ruido eliminado.

[AUDITORIA] Processamento 